# Day 2: Data Preprocessing Pipeline

## 1. Environment setup and data loading
Before we can train any machine learning models, we must transform our raw physics data into a mathematically clean format. For that, we import our preprocessing tools from Scikit-Learn and reload our dataset, ensuring we apply the same anomaly cleaning that we established in our EDA

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Loading data...")
df = pd.read_csv('training.csv')

df.replace(-999.0, np.nan, inplace=True)

print(f"Dataset loaded, shape: {df.shape}")

Loading data...
Dataset loaded, shape: (250000, 33)


## 2. Handling Missing Data: Drop and Impute
Based on our EDA, several jet-specific derived features are missing ~71% of their data because those collisions did not produce secondary jets. We cannot confidently mathematically impute 71% of a column, so we will drop those features entirely.

For features with manageable missing data (like `DER_mass_MMC` at ~15%), we will impute (fill in) the missing values using the median of the column, which is robust against outliers. We also drop `EventId` (an arbitrary identifier) and `Weight` (a Kaggle-specific scoring metric) so they don't confuse the models.

In [9]:
threshold = 0.70
missing_fractions = df.isna().mean()
cols_to_drop = missing_fractions[missing_fractions > threshold].index.tolist()

cols_to_drop.extend(['EventId', 'Weight'])

df_clean = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns : {cols_to_drop}")

for col in df_clean.columns:
    if df_clean[col].isna().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
print(f"Missing values remaining in dataset: {df_clean.isna().sum().sum()}")
print(f"New dataset shape: {df_clean.shape}")


Dropped 9 columns : ['DER_deltaeta_jet_jet', 'DER_mass_jet_jet', 'DER_prodeta_jet_jet', 'DER_lep_eta_centrality', 'PRI_jet_subleading_pt', 'PRI_jet_subleading_eta', 'PRI_jet_subleading_phi', 'EventId', 'Weight']
Missing values remaining in dataset: 0
New dataset shape: (250000, 24)


## 3. Train/Test Split and Feature Scaling

With a clean dataset, we must prepare it for the machine learning algorithms. 

1. **Target Encoding:** We convert our target `Label` from strings (`'s'`, `'b'`) into binary integers (`1`, `0`).
2. **Stratified Split:** We split the data into 80% training and 20% testing. Because we have an imbalanced dataset (65% background, 35% signal), we use `stratify=y` to guarantee that this exact ratio is preserved in both the training and testing sets.
3. **Feature Scaling:** Machine learning models are sensitive to the scale of features. A feature measuring an angle in radians cannot be directly compared to a momentum feature in GeV. We use `StandardScaler` to transform all features to have a mean of 0 and a standard deviation of 1 ($z = \frac{x - \mu}{\sigma}$). Crucially, we fit the scaler *only* on the training data to prevent data leakage.

In [10]:
df_clean['Label'] = df_clean['Label'].map({'s': 1, 'b': 0})
X = df_clean.drop(columns=['Label'])
y = df_clean['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} collisions")
print(f"Testing set: {X_test.shape[0]} collisions")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns)

print("\nFeature scaling complete. All features now have mean=0 and std=1.")
display(X_train.head())


Training set: 200000 collisions
Testing set: 50000 collisions

Feature scaling complete. All features now have mean=0 and std=1.


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,DER_pt_ratio_lep_tau,DER_met_phi_centrality,PRI_tau_pt,PRI_tau_eta,PRI_tau_phi,PRI_lep_pt,PRI_lep_eta,PRI_lep_phi,PRI_met,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_all_pt
0,0.701677,-1.116943,0.687071,-0.744815,1.145981,-0.824094,0.211284,-0.722703,1.288341,0.565559,0.907265,1.192567,-0.185802,1.777816,-0.521572,-0.833133,-0.272873,-0.495335,1.045451,-0.579869,0.967386,-2.107823,0.161659
1,-0.490754,-1.067895,-0.459915,-0.296077,0.632094,-0.713719,-0.538175,0.124293,1.246457,-0.670053,-1.363573,0.971421,-0.460867,-0.675065,-0.997416,-0.594800,-1.195570,-0.598913,0.021864,-0.856159,1.781348,0.652813,-0.377924
2,-0.608436,-0.670638,-0.527506,-0.419194,1.213732,-0.660212,-0.628935,-0.412335,1.291692,-0.647894,-1.361102,0.038974,-0.921982,-0.354231,-1.696312,-0.381912,-1.140418,-0.732019,0.021864,-0.872834,0.249566,0.756800,-0.386106
3,0.861361,0.754527,1.011808,0.128821,0.721577,0.158120,0.045024,-0.725072,-0.033525,0.119352,1.190605,-0.499040,-0.566372,-0.668744,-1.534393,0.118593,-0.085906,0.124274,0.021864,0.227402,0.054389,1.625018,0.153794
4,-0.448505,-0.564099,-0.629815,0.162739,0.437789,-0.696869,0.851508,-0.221613,1.276614,-0.594231,0.887497,-0.938583,-0.677087,0.945701,0.527597,0.398243,0.161176,0.723427,1.045451,0.511811,1.037505,1.621457,1.296649
